# PHASE 6 (Bonus) — Dataset synthétique Togo + Test du modèle
## FRAUDX — Détection de fraude bancaire par IA au Togo

**Objectif :** Générer un dataset de transactions mobile money togolaises et tester le modèle XGBoost pré-entraîné.

---
⚠️ **Avertissement méthodologique :**
Ce jeu de données est **synthétique et exploratoire**. Il ne constitue **pas une validation formelle**
de l'adaptation du modèle au Togo. Les entretiens qualitatifs (Phase 6, option A) restent le socle
principal de la contextualisation. Ce notebook est un **bonus technique** pour illustrer les pistes
de recherche futures.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix

sns.set_style('whitegrid')

---
## 1. Génération du dataset synthétique Togo

In [ ]:
from fraudx.synthetic_data import generate_togo_dataset, generate_togo_dataset_ieee_compatible

# Générer 5000 transactions
df_togo = generate_togo_dataset(n_transactions=5000, fraud_rate=0.035, seed=42)
print(f'\nDimensions : {df_togo.shape}')
print(f'Colonnes : {df_togo.columns.tolist()}')

In [ ]:
df_togo.head(10)

In [ ]:
# === Statistiques descriptives ===
print('=== Statistiques ===')
print(f'Transactions totales : {len(df_togo)}')
print(f'Taux de fraude : {df_togo["isFraud"].mean()*100:.2f}%')
print(f'Montant moyen : {df_togo["montant_cfa"].mean():.0f} FCFA')
print(f'Montant médian : {df_togo["montant_cfa"].median():.0f} FCFA')
print(f'Canal le plus utilisé : {df_togo["canal"].mode()[0]}')
print(f'Opérateur principal : {df_togo["operateur"].mode()[0]}')

In [ ]:
# === Visualisation du dataset Togo ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution des montants
axes[0,0].hist(df_togo['montant_cfa'], bins=50, color='steelblue', edgecolor='black')
axes[0,0].set_xlabel('Montant (FCFA)')
axes[0,0].set_ylabel('Fréquence')
axes[0,0].set_title('Distribution des montants (FCFA)', fontweight='bold')

# Transactions par canal
canal_counts = df_togo['canal'].value_counts()
axes[0,1].bar(canal_counts.index, canal_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[0,1].set_xlabel('Canal')
axes[0,1].set_ylabel('Nombre de transactions')
axes[0,1].set_title('Transactions par canal', fontweight='bold')

# Fraude par canal
fraud_by_canal = df_togo.groupby('canal')['isFraud'].mean() * 100
axes[1,0].bar(fraud_by_canal.index, fraud_by_canal.values, color='coral', edgecolor='black')
axes[1,0].set_xlabel('Canal')
axes[1,0].set_ylabel('Taux de fraude (%)')
axes[1,0].set_title('Taux de fraude par canal', fontweight='bold')

# Taux de fraude par ville
fraud_by_ville = df_togo.groupby('ville')['isFraud'].mean().sort_values() * 100
axes[1,1].barh(fraud_by_ville.index, fraud_by_ville.values, color='mediumseagreen')
axes[1,1].set_xlabel('Taux de fraude (%)')
axes[1,1].set_title('Taux de fraude par ville', fontweight='bold')

plt.suptitle('Figure — Dataset synthétique Togo (mobile money)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Test du modèle XGBoost sur les données Togo

**Objectif :** Appliquer le modèle pré-entraîné (sur IEEE-CIS) au dataset synthétique Togo.
Le modèle n'est PAS réentraîné — on teste sa capacité à généraliser.

In [ ]:
# === Chargement du modèle ===
model = joblib.load('models/xgb_model.pkl')
preprocessor = joblib.load('models/scaler.pkl')
threshold = np.load('models/best_threshold.npy')

print(f'Modèle chargé : {type(model).__name__}')
print(f'Seuil optimal : {threshold:.4f}')

In [ ]:
# === Conversion au format IEEE-CIS ===
df_test = generate_togo_dataset_ieee_compatible(n_transactions=2000, fraud_rate=0.035, seed=123)

X_togo = df_test.drop(columns=['isFraud'], errors='ignore')
y_togo = df_test['isFraud'].values

print(f'X_togo : {X_togo.shape}')

In [ ]:
# === Prédiction ===
from fraudx.preprocessing import FraudPreprocessor, FeatureEngineer

prep = FraudPreprocessor()
prep.load_artifacts()

X_processed = FeatureEngineer.add_temporal_features(X_togo)
X_processed = FeatureEngineer.add_amount_features(X_processed)
X_processed = FeatureEngineer.add_behavioral_features(X_processed)

try:
    X_final = prep.transform(X_processed)
    print(f'✅ Prétraitement réussi : {X_final.shape}')
except Exception as e:
    print(f'⚠️  Erreur de prétraitement : {e}')
    # Fallback : utiliser les colonnes communes
    common_cols = [c for c in prep.scaler.feature_names_in_ if c in X_processed.columns]
    missing = [c for c in prep.scaler.feature_names_in_ if c not in X_processed.columns]
    print(f'   Colonnes communes : {len(common_cols)}/{len(prep.scaler.feature_names_in_)}')
    print(f'   Colonnes manquantes : {missing[:10]}...')
    for c in missing:
        X_processed[c] = 0.0
    X_final = prep.transform(X_processed)

In [ ]:
# === Évaluation ===
y_probs = model.predict_proba(X_final)[:, 1]
y_pred = (y_probs >= threshold).astype(int)

print('=== Performance sur données synthétiques Togo ===')
print(f'F1-Score    : {f1_score(y_togo, y_pred):.4f}')
print(f'Recall      : {recall_score(y_togo, y_pred):.4f}')
print(f'Precision   : {precision_score(y_togo, y_pred):.4f}')

cm = confusion_matrix(y_togo, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f'\nMatrice de confusion :')
print(f'  TN={tn}, FP={fp}, FN={fn}, TP={tp}')

In [ ]:
# === Analyse par canal ===
results = X_togo.copy()
results['y_true'] = y_togo
results['y_pred'] = y_pred

if 'ProductCD' in results.columns:
    canal_mapping = {'USSD': 'USSD', 'APP': 'APP', 'AGENT': 'AGENT', 'WEB': 'WEB'}
    results['canal'] = results['ProductCD'].map(canal_mapping).fillna('AUTRE')
    print('=== Performance par canal ===')
    for canal in results['canal'].unique():
        subset = results[results['canal'] == canal]
        if len(subset) > 0 and subset['y_true'].sum() > 0:
            f1_canal = f1_score(subset['y_true'], subset['y_pred'])
            recall_canal = recall_score(subset['y_true'], subset['y_pred'])
            print(f'  {canal:12s} | F1={f1_canal:.3f} | Recall={recall_canal:.3f} | N={len(subset)}')

---
## 3. Interprétation pour le mémoire

**Formulation prudente recommandée :**

> *"Ce test exploratoire sur données synthétiques suggère que le modèle, bien que non réentraîné,
> capture en partie certains patterns de fraude pertinents pour le Togo (SIM swap, montant élevé,
> transactions inhabituelles), mais peine sur les canaux USSD et Agent spécifiques au mobile money.
> Ces résultats ne constituent pas une validation formelle de HS2 ; les entretiens qualitatifs
> menés auprès des responsables bancaires togolais en sont le socle principal."*

### Pistes d'amélioration identifiées :
1. **Réentraîner le modèle** sur un mélange IEEE-CIS + données togolaises synthétiques
2. **Ajouter des features spécifiques** : fréquence des recharges, canaux USSD, types d'agents
3. **Partenariat** avec un opérateur mobile money (TogoCom Cash, Moov Money) pour données réelles
4. **Test A/B** en sandbox réglementaire BCEAO